In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
   !pip install -q "optimum[onnxruntime]" rouge-score sentencepiece

In [ ]:
!pip uninstall -y diffusers
!pip install -q "optimum[onnxruntime]" rouge-score sentencepiece

In [ ]:
import os
M = "/kaggle/input/datasets/tanushkalokegaonkar/models/intent-classifier-int8/intent-classifier-int8"
print(os.listdir(M))
D = "/kaggle/input/datasets/prishabhkumar/email-intelligence-processed-data/test"
print(os.listdir(D))

In [ ]:
import pandas as pd
for f in ["enron_test.csv", "spamassassin_test.csv"]:
    df = pd.read_csv(f"{D}/{f}")
    print(f, df.shape)
    print(df.columns.tolist())
    print(df.head(2).T, "\n")

# Find the summarisation files

In [ ]:
import pandas as pd
CSV_PATH = "/kaggle/input/datasets/marawanxmamdouh/email-thread-summary-dataset/CSV/email_thread_summaries.csv"
d = pd.read_csv(CSV_PATH)
print(d.shape)
print(d.columns.tolist())
print(d.head(2).T)

In [ ]:
import os
R = "/kaggle/input/datasets/marawanxmamdouh/email-thread-summary-dataset"
for root, _, files in os.walk(R):
    for f in files:
        print(os.path.join(root, f))

In [ ]:
import glob, pandas as pd
for f in glob.glob(R + "/**/*.csv", recursive=True):
    x = pd.read_csv(f, nrows=3)
    print(f, x.columns.tolist(), "\n")

# Build one input per thread (adjust names to match 3b)

In [ ]:
import pandas as pd

R = "/kaggle/input/datasets/marawanxmamdouh/email-thread-summary-dataset/CSV"
summ  = pd.read_csv(f"{R}/email_thread_summaries.csv")
mails = pd.read_csv(f"{R}/email_thread_details.csv")

mails["timestamp"] = pd.to_datetime(mails["timestamp"], errors="coerce")
mails = mails.dropna(subset=["body"]).sort_values(["thread_id", "timestamp"])

threads = (mails.groupby("thread_id")["body"]
                .apply(lambda s: "\n\n".join(s.astype(str)))
                .reset_index()
                .rename(columns={"body": "email_body"}))

data = threads.merge(summ, on="thread_id").dropna(subset=["email_body", "summary"])
print(data.shape)
print(data.iloc[0].email_body[:500])
print("REF:", data.iloc[0].summary)

# Config and load model

In [ ]:
import glob, os
from transformers import AutoTokenizer
from optimum.onnxruntime import ORTModelForSeq2SeqLM

M = "/kaggle/input/datasets/tanushkalokegaonkar/models/t5_email_small_onnx_int8/t5_email_small_onnx_int8"
TEXT_COL, REF_COL = "email_body", "summary"
N_SAMPLES = 20
PREFIX = "summarize: "   # must match training
MAX_IN = 512             # must match training
MAX_OUT = 128
BEAMS = 4

test_df = data.sample(min(N_SAMPLES, len(data)), random_state=42)
tok = AutoTokenizer.from_pretrained(M)

ENC = "encoder_model_quantized.onnx"
DEC = "decoder_model_merged_quantized.onnx"

try:
    # Attempt A: merged decoder used for both slots (keeps KV-cache, fastest)
    model = ORTModelForSeq2SeqLM.from_pretrained(
        M,
        encoder_file_name=ENC,
        decoder_file_name=DEC,
        decoder_with_past_file_name=DEC,
        provider="CPUExecutionProvider",
    )
    print("loaded with cache (merged decoder)")
except Exception as e:
    print("Attempt A failed:", repr(e)[:300])
    # Attempt B: no cache (slower, same outputs)
    model = ORTModelForSeq2SeqLM.from_pretrained(
        M,
        encoder_file_name=ENC,
        decoder_file_name=DEC,
        use_cache=False,
        provider="CPUExecutionProvider",
    )
    print("loaded without cache")

print("rows:", len(test_df))

# Smoke test

In [ ]:
text = str(test_df[TEXT_COL].iloc[0])
inp = tok(PREFIX + text, return_tensors="pt", truncation=True, max_length=MAX_IN)
out = model.generate(**inp, max_length=MAX_OUT, num_beams=BEAMS)
print("PRED:", tok.decode(out[0], skip_special_tokens=True))
print("REF: ", test_df[REF_COL].iloc[0])

# Compute ROUGE 

In [ ]:
from tqdm.auto import tqdm
from rouge_score import rouge_scorer

keys = ["rouge1", "rouge2", "rougeL", "rougeLsum"]
scorer = rouge_scorer.RougeScorer(keys, use_stemmer=True)
scores = {k: [] for k in keys}
preds = []

for text, ref in tqdm(zip(test_df[TEXT_COL], test_df[REF_COL]), total=len(test_df)):
    inp = tok(PREFIX + str(text), return_tensors="pt", truncation=True, max_length=MAX_IN)
    out = model.generate(**inp, max_length=MAX_OUT, num_beams=BEAMS)
    pred = tok.decode(out[0], skip_special_tokens=True)
    preds.append(pred)
    s = scorer.score(str(ref), pred)      # (reference, prediction)
    for k in keys:
        scores[k].append(s[k].fmeasure)

for k in keys:
    print(f"{k}: {sum(scores[k]) / len(scores[k]) * 100:.2f}")

# # INTENT

# Inspect the intent model folder

In [1]:
import os, json
I = "/kaggle/input/datasets/tanushkalokegaonkar/models/intent-classifier-int8/intent-classifier-int8"
for root, _, files in os.walk(I):
    for f in files:
        print(os.path.join(root, f))

cfg = glob.glob(I + "/**/config.json", recursive=True) if (glob := __import__("glob")) else []
if cfg:
    c = json.load(open(cfg[0]))
    print(c.get("model_type"), c.get("id2label"), c.get("architectures"))

/kaggle/input/datasets/tanushkalokegaonkar/models/intent-classifier-int8/intent-classifier-int8/config.json
/kaggle/input/datasets/tanushkalokegaonkar/models/intent-classifier-int8/intent-classifier-int8/model_quantized.pt
/kaggle/input/datasets/tanushkalokegaonkar/models/intent-classifier-int8/intent-classifier-int8/tokenizer.json
/kaggle/input/datasets/tanushkalokegaonkar/models/intent-classifier-int8/intent-classifier-int8/tokenizer_config.json
distilbert {'0': 'REQUEST', '1': 'FOLLOW_UP', '2': 'INFORMATION', '3': 'ACKNOWLEDGEMENT', '4': 'COMPLAINT', '5': 'INVITATION'} ['DistilBertForSequenceClassification']


# Load the model

In [2]:
import transformers, torch
print(transformers.__version__, torch.__version__)

5.20.1 2.11.0+cpu


In [ ]:
!pip install -q -U transformers huggingface_hub

In [3]:
import torch
from transformers import AutoTokenizer, AutoConfig

I = "/kaggle/input/datasets/tanushkalokegaonkar/models/intent-classifier-int8/intent-classifier-int8"
tok = AutoTokenizer.from_pretrained(I)
cfg = AutoConfig.from_pretrained(I)

m = torch.load(f"{I}/model_quantized.pt", map_location="cpu", weights_only=False)
m.eval()
id2label = {int(k): v for k, v in cfg.id2label.items()}
print(type(m).__name__, id2label)

# Save the weights so later sessions don't need the upgrade:
torch.save(m.state_dict(), "/kaggle/working/intent_state_dict.pt")

/usr/local/lib/python3.13/dist-packages/torch/_utils.py:455: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  device=storage.device,


DistilBertForSequenceClassification {0: 'REQUEST', 1: 'FOLLOW_UP', 2: 'INFORMATION', 3: 'ACKNOWLEDGEMENT', 4: 'COMPLAINT', 5: 'INVITATION'}


# Sanity test 

In [4]:
enc = tok(["Can you send me the report by Friday?", "Thanks, got it."],
          return_tensors="pt", padding=True, truncation=True, max_length=256)
with torch.inference_mode():
    print([id2label[i] for i in m(**enc).logits.argmax(-1).tolist()])

['REQUEST', 'ACKNOWLEDGEMENT']


# Find labelled data

In [5]:
import pandas as pd

CSV = "/kaggle/input/datasets/prishabhkumar/processed-intent-data/test/intent_test.csv"
TEXT_COL, LABEL_COL = "cleaned_body", "intent_label"
N = 2000
MAX_IN = 256                      # must match training

df = pd.read_csv(CSV).dropna(subset=[TEXT_COL, LABEL_COL, "intent_name"])
df[LABEL_COL] = df[LABEL_COL].astype(int)

# The dataset's ids must match the model's ids, or every score will be wrong
print(df.groupby([LABEL_COL, "intent_name"]).size())
print(id2label)

df = df.sample(min(N, len(df)), random_state=42)
print(df["intent_name"].value_counts())
print(df["confidence_score"].describe())

intent_label  intent_name    
0             REQUEST            1692
1             FOLLOW_UP           330
2             INFORMATION        1376
3             ACKNOWLEDGEMENT    6147
4             COMPLAINT           115
5             INVITATION          259
dtype: int64
{0: 'REQUEST', 1: 'FOLLOW_UP', 2: 'INFORMATION', 3: 'ACKNOWLEDGEMENT', 4: 'COMPLAINT', 5: 'INVITATION'}
intent_name
ACKNOWLEDGEMENT    1239
REQUEST             328
INFORMATION         286
FOLLOW_UP            69
INVITATION           56
COMPLAINT            22
Name: count, dtype: int64
count    2000.000000
mean        2.538500
std         1.128785
min         2.000000
25%         2.000000
50%         2.000000
75%         3.000000
max        17.000000
Name: confidence_score, dtype: float64


In [6]:
df = pd.read_csv(CSV).dropna(subset=[TEXT_COL, LABEL_COL, "intent_name"])
df[LABEL_COL] = df[LABEL_COL].astype(int)
print(len(df))

9919


# Predict and compute metrics

In [7]:
import numpy as np
from tqdm.auto import tqdm
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             classification_report, confusion_matrix)

texts = df[TEXT_COL].astype(str).tolist()
preds = []
BATCH = 32
with torch.inference_mode():
    for i in tqdm(range(0, len(texts), BATCH)):
        enc = tok(texts[i:i+BATCH], return_tensors="pt", truncation=True,
                  max_length=MAX_IN, padding=True)
        preds += m(**enc).logits.argmax(-1).tolist()

y_true, y_pred = df[LABEL_COL].values, np.array(preds)
names = [id2label[i] for i in range(len(id2label))]

print("accuracy:", round(accuracy_score(y_true, y_pred) * 100, 2))
for avg in ["macro", "weighted"]:
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, average=avg, zero_division=0)
    print(f"{avg}: P={p*100:.2f} R={r*100:.2f} F1={f*100:.2f}")
print(classification_report(y_true, y_pred, labels=list(range(len(names))),
                            target_names=names, zero_division=0))
print(confusion_matrix(y_true, y_pred, labels=list(range(len(names)))))

  0%|          | 0/310 [00:00<?, ?it/s]

accuracy: 90.25
macro: P=86.93 R=83.82 F1=85.25
weighted: P=90.31 R=90.25 F1=90.21
                 precision    recall  f1-score   support

        REQUEST       0.91      0.83      0.86      1692
      FOLLOW_UP       0.87      0.77      0.82       330
    INFORMATION       0.82      0.88      0.85      1376
ACKNOWLEDGEMENT       0.93      0.94      0.93      6147
      COMPLAINT       0.79      0.77      0.78       115
     INVITATION       0.91      0.85      0.88       259

       accuracy                           0.90      9919
      macro avg       0.87      0.84      0.85      9919
   weighted avg       0.90      0.90      0.90      9919

[[1396   12   58  219    3    4]
 [  10  254   10   56    0    0]
 [  18    2 1209  141    5    1]
 [ 108   25  197 5785   15   17]
 [   1    0    2   24   88    0]
 [   9    0    6   24    0  220]]


# Score by labeller confidence

In [8]:
import numpy as np
for th in [2, 3, 4, 5]:
    mask = (df["confidence_score"].values >= th)
    if mask.sum() < 50:
        continue
    p, r, f, _ = precision_recall_fscore_support(y_true[mask], y_pred[mask], average="macro", zero_division=0)
    print(f"conf>={th}: rows={mask.sum()}  acc={accuracy_score(y_true[mask], y_pred[mask])*100:.2f}  macro-F1={f*100:.2f}")

conf>=2: rows=9919  acc=90.25  macro-F1=85.25
conf>=3: rows=3046  acc=89.26  macro-F1=81.48
conf>=4: rows=1195  acc=89.87  macro-F1=77.09
conf>=5: rows=522  acc=87.93  macro-F1=75.09
